# AgentOps Lab 03 - Rebuild the agent using OpenAI Agents SDK

In Notebook 1, you built the loop yourself. In Notebook 2, you learned when not to use an agent. Now you rebuild the incident investigator using a framework-shaped design.

The OpenAI Agents SDK is useful when you want a runtime to manage turns, tool execution, guardrails, handoffs, sessions, and tracing. The lesson is subtle but important: frameworks do not remove the agent loop. They package it.


## What moves into the framework?

```mermaid
flowchart LR
    A["Manual loop"] --> B["Application owns messages"]
    A --> C["Application dispatches tools"]
    A --> D["Application records trace"]
    E["Agents SDK"] --> F["Runner manages turns"]
    E --> G["Function tools expose schemas"]
    E --> H["Tracing records model and tool spans"]
    E --> I["Sessions preserve working context"]
```

You still own the product boundary: which tools exist, what they are allowed to do, which actions require approval, what evidence is sufficient, and what counts as safe completion.


## Real SDK shape

The real implementation looks like this. This cell is shown as reference because it requires `openai-agents` and `OPENAI_API_KEY`.

```python
from agents import Agent, Runner, function_tool

@function_tool
def get_service_status(service: str) -> dict:
    ...

@function_tool
def search_incidents(query: str) -> list:
    ...

@function_tool
def get_runbook(service: str) -> str:
    ...

incident_agent = Agent(
    name="Incident Investigator",
    instructions="""
    Investigate operational incidents.
    Always gather evidence before diagnosing a problem.
    Use the minimum number of tools required.
    Never execute remediation actions.
    """,
    tools=[get_service_status, search_incidents, get_runbook],
)

result = await Runner.run(incident_agent, "European users report checkout failures.")
print(result.final_output)
```

The SDK docs describe Agents as models equipped with instructions and tools; function tools turn Python functions into tools with schema generation; sessions carry working context; tracing records model calls, tool calls, guardrails, handoffs, and custom events.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
sys.path.insert(0, str(repo_root / "labs"))

from agentops_lab.agents_sdk_rebuild import OfflineAgentsSDKRuntime, TOOLS, compare_manual_and_framework


## Run the offline framework-shaped version

This teaching double keeps the notebook runnable without credentials. It mirrors the responsibilities you would inspect in a framework run: session start, model planning, tool schema/dispatch, guardrail outcome, and final response.


In [ ]:
comparison = compare_manual_and_framework()
comparison["framework"]["final_output"]


In [ ]:
print("Manual owns:")
for item in comparison["manual"]["owned_by_application"]:
    print("-", item)

print("\nFramework owns:")
for item in comparison["framework"]["framework_owns"]:
    print("-", item)


## Inspect the trace

A trace should answer: what did the agent decide, which tools were exposed, which calls were made, what observations came back, which guardrails passed, and why did the run stop?


In [ ]:
for event in comparison["framework"]["trace"]:
    print(f"{event['kind']:12} {event['name']}")


## Optional real SDK experiment

Install `openai-agents`, set `OPENAI_API_KEY`, and port the deterministic tools into `@function_tool` functions. Keep the tools read-only for this notebook. Then compare the real trace against the offline trace above.

Questions to answer:

- Which parts of the manual loop disappeared from your code?
- Which safety decisions still live in your application?
- Did the framework reduce code, or did it move the code into configuration?
- What trace span would you inspect first if the agent overused tools?


## Takeaway

The SDK packages the loop, schemas, dispatch, messages, sessions, and tracing. It does not choose your product boundary for you. You still design tools, permissions, evidence rules, budgets, approval gates, and evaluation.

References: [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/), [Agents SDK tools](https://openai.github.io/openai-agents-python/tools/), [Agents SDK tracing](https://openai.github.io/openai-agents-python/tracing/), and [Agents SDK sessions](https://openai.github.io/openai-agents-python/sessions/).
